# 02 - Energy Demand Forecasting
## XGBoost & LightGBM Models with MLflow Tracking

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.terna_loader import load_demand_data
from src.data.feature_engineering import build_feature_matrix, prepare_train_test, get_feature_columns
from src.models.ml_models import train_xgboost_forecast, train_lightgbm_forecast, cross_validate_forecast
from src.models.explainability import compute_shap_values, get_feature_importance
from src.monitoring.mlops import setup_mlflow, log_experiment

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## Load and Prepare Data

In [ ]:
demand = load_demand_data()
print(f'Raw demand shape: {demand.shape}')

df = build_feature_matrix(demand, target_col='demand_mw')
df = df.dropna()
print(f'Feature matrix shape: {df.shape}')
print(f'Features: {df.shape[1] - 1}')
df.head()

In [ ]:
feature_cols = get_feature_columns(df, target_col='demand_mw')
print(f'Number of features: {len(feature_cols)}')

train, test = prepare_train_test(df, test_ratio=0.2)
print(f'Train: {train.shape}, Test: {test.shape}')

X_train = train[feature_cols]
y_train = train['demand_mw']
X_test = test[feature_cols]
y_test = test['demand_mw']

## XGBoost Forecasting

In [ ]:
xgb_model, xgb_pred, xgb_metrics = train_xgboost_forecast(X_train, y_train, X_test, y_test)
print('XGBoost Metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v:,.4f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(test.index, y_test.values, label='Actual', alpha=0.7)
axes[0].plot(test.index, xgb_pred, label='XGBoost Prediction', alpha=0.7)
axes[0].set_title('XGBoost - Actual vs Predicted')
axes[0].set_ylabel('Demand (MW)')
axes[0].legend()

errors = y_test.values - xgb_pred
axes[1].plot(test.index, errors, color='red', alpha=0.5, linewidth=0.5)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_title('Prediction Error')
axes[1].set_ylabel('Error (MW)')

plt.tight_layout()
plt.show()

## LightGBM Forecasting

In [ ]:
lgb_model, lgb_pred, lgb_metrics = train_lightgbm_forecast(X_train, y_train, X_test, y_test)
print('LightGBM Metrics:')
for k, v in lgb_metrics.items():
    print(f'  {k}: {v:,.4f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(test.index, y_test.values, label='Actual', alpha=0.7)
axes[0].plot(test.index, lgb_pred, label='LightGBM Prediction', alpha=0.7)
axes[0].set_title('LightGBM - Actual vs Predicted')
axes[0].set_ylabel('Demand (MW)')
axes[0].legend()

errors = y_test.values - lgb_pred
axes[1].plot(test.index, errors, color='red', alpha=0.5, linewidth=0.5)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_title('Prediction Error')
axes[1].set_ylabel('Error (MW)')

plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'XGBoost': xgb_metrics,
    'LightGBM': lgb_metrics,
}).T
print(comparison.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
comparison[['mae', 'rmse']].plot(kind='bar', ax=ax)
ax.set_title('Model Comparison - Error Metrics')
ax.set_ylabel('Error (MW)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## SHAP Explainability

In [ ]:
explainer, shap_values = compute_shap_values(xgb_model, X_test)
feature_importance = get_feature_importance(shap_values, feature_cols)

print('Top 15 Features by SHAP Importance:')
print(feature_importance.head(15).to_string(index=False))

In [ ]:
top_n = 15
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(top_n)
ax.barh(range(top_n), top_features['importance'].values[::-1])
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'].values[::-1])
ax.set_title(f'Top {top_n} Features - SHAP Importance')
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.show()

In [ ]:
sample_idx = 0
explanation = pd.DataFrame({
    'feature': feature_cols,
    'value': X_test.iloc[sample_idx].values,
    'shap_value': shap_values[sample_idx],
}).sort_values('shap_value', key=abs, ascending=False)

print(f'Explanation for sample at {X_test.index[sample_idx]}:')
print(f'Actual: {y_test.iloc[sample_idx]:,.0f} MW')
print(f'Predicted: {xgb_pred[sample_idx]:,.0f} MW')
print(f'\nTop contributing features:')
print(explanation.head(10).to_string(index=False))

## Cross-Validation

In [ ]:
from lightgbm import LGBMRegressor

cv_model = LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, verbose=-1)
cv_results = cross_validate_forecast(cv_model, df[feature_cols], df['demand_mw'], n_splits=5)

print('Cross-Validation Results:')
print(cv_results.mean().to_string())

## MLflow Logging

In [ ]:
mlflow = setup_mlflow()

log_experiment(
    model_name='XGBoost_Demand_Forecast',
    model=xgb_model,
    metrics=xgb_metrics,
    params=xgb_model.get_params(),
    feature_importance=feature_importance,
    tags={'task': 'forecasting', 'data': 'terna_demand', 'target': 'demand_mw'},
)

log_experiment(
    model_name='LightGBM_Demand_Forecast',
    model=lgb_model,
    metrics=lgb_metrics,
    params=lgb_model.get_params(),
    feature_importance=feature_importance,
    tags={'task': 'forecasting', 'data': 'terna_demand', 'target': 'demand_mw'},
)

print('Experiments logged to MLflow.')

In [ ]:
print('Notebook 02 complete. Forecasting models trained and evaluated.')